<a href="https://colab.research.google.com/github/im1xd/SpamDetection/blob/main/SPAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 1) إعداد المكتبات وقراءة البيانات

In [ ]:
!pip install scikit-learn gradio --quiet

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import gradio as gr


In [ ]:
df = pd.read_csv("/content/spam.csv", encoding="latin-1")

df = df[['v1', 'v2']].copy()
df = df.rename(columns={'v1': 'label', 'v2': 'message'})

print(df.head())
print(df['label'].value_counts())


  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
label
ham     4825
spam     747
Name: count, dtype: int64


 2) تجهيز البيانات + تقسيمها

In [ ]:
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

X = df['message']
y = df['label_num']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

len(X_train), len(X_test)


(4457, 1115)

In [ ]:
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

X = df['message']
y = df['label_num']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

len(X_train), len(X_test)


(4457, 1115)

# 3) TF-IDF + تدريب نموذج Naive Bayes (مع ضبط alpha)

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

model = MultinomialNB(alpha=0.1)
model.fit(X_train_tfidf, y_train)

print("Model classes_:", model.classes_)


Model classes_: [0 1]


# 4) تقييم النموذج

In [ ]:
y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred, target_names=['ham', 'spam']))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.9874439461883409

Classification report:
               precision    recall  f1-score   support

         ham       0.99      1.00      0.99       966
        spam       0.99      0.91      0.95       149

    accuracy                           0.99      1115
   macro avg       0.99      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115


Confusion matrix:
 [[965   1]
 [ 13 136]]


# 5) اختبار يدوي

In [ ]:
THRESHOLD = 0.3

sample_msgs = [
    "Congratulations! You have won a free ticket to Bahamas. Call now!",
    "Hey, are we still meeting tomorrow?",
    "URGENT! Your account will be suspended unless you update your data at this link."
]

sample_tfidf = vectorizer.transform(sample_msgs)
sample_proba = model.predict_proba(sample_tfidf)

for msg, proba in zip(sample_msgs, sample_proba):
    ham_prob = proba[0]
    spam_prob = proba[1]

    label = "spam" if spam_prob >= THRESHOLD else "ham"

    print("Message:", msg)
    print(f"→ Predicted label: {label}")
    print(f"   P(ham)={ham_prob:.3f} | P(spam)={spam_prob:.3f}")
    print("-" * 60)


Message: Congratulations! You have won a free ticket to Bahamas. Call now!
→ Predicted label: spam
   P(ham)=0.193 | P(spam)=0.807
------------------------------------------------------------
Message: Hey, are we still meeting tomorrow?
→ Predicted label: ham
   P(ham)=0.999 | P(spam)=0.001
------------------------------------------------------------
Message: URGENT! Your account will be suspended unless you update your data at this link.
→ Predicted label: spam
   P(ham)=0.620 | P(spam)=0.380
------------------------------------------------------------


# 6) بناء واجهة Gradio

In [ ]:
THRESHOLD = 0.3
def classify_message(text):
    if not text or not text.strip():
        return "الرجاء كتابة رسالة أولاً."

    X = vectorizer.transform([text])
    proba = model.predict_proba(X)[0]

    ham_prob = proba[0]
    spam_prob = proba[1]

    if spam_prob >= THRESHOLD:
        label = "رسالة مزعجة (Spam)"
    else:
        label = "ليست مزعجة (Ham)"

    return f"""التصنيف: {label}
احتمال أن تكون Spam: {spam_prob:.2f}
احتمال أن تكون Ham: {ham_prob:.2f}"""

demo = gr.Interface(
    fn=classify_message,
    inputs=gr.Textbox(
        lines=4,
        placeholder="اكتب هنا نص الرسالة (بالإنجليزية يفضّل)...",
        label="نص الرسالة"
    ),
    outputs=gr.Textbox(label="نتيجة التصنيف"),
    title="نموذج كشف الرسائل المزعجة (Spam Filter - Naive Bayes)",
    description="أدخل نص رسالة SMS (بالإنجليزية) وسيقوم النموذج بتصنيفها إلى Spam أو Ham."
)

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://52e20744c1b3e41f0e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
